# Problem Set 10

[PSet 10](pset10.pdf)


In [1]:
import itertools

import sympy as sp
import sympy.physics.units as spu
import sympy.physics.mechanics as spm
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

HALF = sp.S.Half
PI = sp.pi

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> spm.ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return spm.ReferenceFrame(
        frame,
        latexs=(
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{x}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{y}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{z}}}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> spm.ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

In [3]:
def simplify_square_symbol(value, rationaliser):
    num, den = value.as_numer_denom()
    num = (num / rationaliser**2).simplify().collect([g])
    den = (den / rationaliser**2).simplify()
    return num / den

def simplify_by_symbol(value, symbol):
    num, den = value.as_numer_denom()
    num = (num / symbol).simplify().collect([g])
    den = (den / symbol).simplify()
    return num / den

def flatten_list(nested_list):
    return [item for sublist in nested_list for item in sublist]

## Pivot

In physics, a pivot (also called a fulcrum or axis of rotation) is a fixed
point or shaft around which an object rotates, turns, or balances. The rest
of the object moves in a circular path around it while the pivot remains
stationary.

### How a Pivot Works in Mechanics

Pivots are essential for understanding rotational motion and simple
machines. Here are some of their main functions:

- **Torque and leverage:** A force applied away from a pivot creates a
  turning effect called torque (or a moment). The farther from the pivot
  you push, the easier it is to rotate the object.
- **Levers:** In a lever system, the pivot is the fulcrum. Moving it closer
  to a heavy object can let a small force on the other side lift a heavier
  load.
- **Equilibrium and balance:** An object is balanced when clockwise and
  counterclockwise torques about the pivot are equal.

### Everyday Examples

- **Seesaw:** The central support is the pivot.
- **Door:** The hinges form a vertical pivot axis, allowing the door to
  swing open and shut.
- **Scissors:** The screw or rivet holding the blades together is the pivot.
- **Wrench:** When tightening a bolt, its center is the pivot around which
  the wrench turns.

Would you like to explore how to calculate torque around a pivot, or how
moving the pivot changes the mechanical advantage of different levers?


Let's look at calculating torque around a pivot and how moving the pivot
changes mechanical advantage.

1. How to Calculate Torque

Torque ($\tau$) measures how effectively a force causes an object to rotate
around a pivot. Calculate it using:
$\tau = F\times d\times\sin(\theta)$

- **$F$ (force):** The size of the push or pull, measured in newtons
  ($\text{N}$).
- **$d$ (distance/lever arm):** The distance from the pivot to where the
  force is applied, measured in meters ($\text{m}$).
- **$\theta$ (angle):** The angle between the force and the lever arm.
  Pushing straight down gives $\theta=90^\circ$, so $\sin(90^\circ)=1$.
  The torque formula then simplifies to $\tau=F\times d$.

Quick Example

Imagine using a $0.25\text{ m}$ wrench to turn a bolt (the pivot). Push
down with $40\text{ N}$ at a $90^\circ$ angle. The torque is:

$\tau=40\text{ N}\times0.25\text{ m}=10\text{ N}\cdot\text{m}$

A $0.5\text{ m}$ wrench with the same force doubles the torque to
$20\text{ N}\cdot\text{m}$.

2. How Moving the Pivot Changes Mechanical Advantage

Mechanical advantage (MA) measures how much a lever multiplies the input
force. It is the effort-arm distance divided by the load-arm distance:
$\text{MA}=\frac{d_{\text{effort}}}{d_{\text{load}}}$
- **Pivot closer to the load:** This increases $d_{\text{effort}}$ and
  decreases $d_{\text{load}}$, giving $\text{MA}>1$. Less force is needed
  to lift a heavy load, but the effort end moves farther.
- **Pivot at the center:** Equal distances give $\text{MA}=1$. There is no
  force multiplication, but the force direction changes, as on a seesaw.
- **Pivot closer to the effort:** This shortens $d_{\text{effort}}$ relative
  to $d_{\text{load}}$, giving $\text{MA}<1$. More force is needed, but the
  load moves faster and farther, as with a broom or fishing rod.

### Comparison of the Three Lever Classes

Pivot position relative to the effort and load determines the lever class:

- **Class 1:** Pivot between effort and load. MA can be $>1$, $=1$, or
  $<1$. Examples: seesaw, crowbar, and scissors.
- **Class 2:** Load between pivot and effort. Always $\text{MA}>1$; it
  multiplies force. Examples: wheelbarrow and nutcracker.
- **Class 3:** Effort between pivot and load. Always $\text{MA}<1$; it
  multiplies speed and distance. Examples: tweezers, forearm, and broom.

Would you like to try a torque practice problem, or explore how rotational
inertia affects how fast an object spins around a pivot?


## Bug Walking on Pivoted Ring

A ring of radius $R$ and mass $m_1$ lies on its side on a frictionless table. It is pivoted to the table at its rim. A bug of mass $m_2$ walks on the ring at a constant speed $v$ relative to the ring, starting at the pivot when the ring is initially at rest. Take $\hat{k}$ to point out of the page.

<img src="../figures/PS11-01-Bug-on-Ring.png" alt="Bug on Ring" style="width: 268.5px; max-width: 100%;">

(a) What is the angular velocity of the ring when the bug is halfway around? Express your answer in terms of some or all of the following: 


$$m_1, m_2, v, R, \hat{k}$$

(b) What is the angular velocity of the ring when the bug is back at the pivot? 

Express your answer in terms of some or all of the following: 

$$m_1, m_2, v, R, \hat{k}$$
